# Statistics & Machine Learning Cheat Sheet: Week 1 Practical Lecture
> Author: Dr. Ghadeer Mobasher

> **Important Note about Exercise Style:** 
- The exercises are designed so that team members can work on solving **Modules A and B** independently. We will then have a short break, after which the same approach will be followed for **Modules C and D**, which can also be solved independently. 
- Each module will include starter code that is heavily commented. Your role will be to complete the **coding exercises and answer the reflection questions**. 
- The modules will be made available shortly before the practical lecture.
### Setup: Main Libraries & Dataset


> We will be working with **[Pima Indians Diabetes](https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv) Dataset (768 rows × 9 columns) **. It would be great if you download the dataset and learn about it's main features. 
```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # statistical data visualization
from scipy import stats # open-source software for mathematics, science, and engineering

df = pd.read_csv("diabetes.csv")
```

> Have a quick overview of [seaborn](https://seaborn.pydata.org/tutorial.html) and [SciPy](https://docs.scipy.org/doc/) documentations. 

---

## Module A: Data Inspection & Quality (Student 1 and Student 2)
*Tasks 1 – 3*

### Core libraries
| Library | What you use it for |
|---|---|
| `pandas` | Loading, inspecting, filtering, cleaning data |
| `numpy` | `np.nan`, the missing-value placeholder |

### Key syntax

```python
# --- Inspect ---
df.shape                        # (rows, columns)
df.dtypes                       # type of each column
df.describe()                   # mean, std, min, quartiles, max
df['Outcome'].value_counts()    # how many 0s and 1s?

# --- Spot hidden missing values (zeros) ---
(df == 0).sum()                 # count zeros per column

# --- Replace impossible zeros with NaN ---
cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols] = df[cols].replace(0, np.nan)

# --- After cleaning ---
df.isnull().sum()               # count NaNs
df.mean()                       # mean now ignores NaN automatically
df.median()

# --- Handle NaNs (pick one strategy) ---
df.dropna()                     # remove rows with any NaN
df.fillna(df.median())          # fill NaN with column median
```

> **Why median for imputation?** Median ignores extreme values, **safer than mean for skewed columns like Insulin.**

---

## Module B: Descriptive Statistics & Visualisation (Student 3 and Students (4-5))
*Tasks 4 – 5*

### Core libraries
| Library | What you use it for |
|---|---|
| `pandas` | `.groupby()`, `.mean()`, `.skew()` |
| `matplotlib` | Low-level plots: histograms, vertical lines |
| `seaborn` | High-level plots: boxplots, pairplots, heatmaps |

### Key syntax

```python
# --- Central tendency & spread ---
df['Glucose'].mean()
df['Glucose'].median()
df['Glucose'].mode()[0]         # [0] because mode() returns a Series
df['Glucose'].std()
df['Glucose'].skew()            # > 0 = right tail, < 0 = left tail

# --- Group by diabetes status ---
df.groupby('Outcome')['Glucose'].mean()
df.groupby('Outcome').describe()

# --- Histogram ---
df['Glucose'].plot.hist(bins=30)
plt.xlabel('Glucose'); plt.title('Glucose Distribution')
plt.show()

# --- Boxplot by group ---
sns.boxplot(data=df, x='Outcome', y='BMI')
plt.show()

# --- Mark mean & median on a plot ---
plt.axvline(df['Insulin'].mean(),   color='red',  linestyle='--', label='Mean')
plt.axvline(df['Insulin'].median(), color='blue', linestyle=':',  label='Median')
plt.legend()

# --- Pairplot ---
sns.pairplot(df[['Glucose','BMI','Insulin','Outcome']], hue='Outcome')
plt.show()

# --- Correlation heatmap ---
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='RdBu_r', center=0)
plt.show()
```

> **Skewness rule of thumb:** 
- |skew| < 0.5 ≈ symmetric 
-  0.5 >= |skew|<= 1 = moderate 
- |skew| > 1 = high skew
---


## Module C: Fundamentals of Probability (Student 3 and Students (4-5))
*Tasks 6 – 7*

### Core libraries
| Library | What you use it for |
|---|---|
| `pandas` | Boolean filtering to count events |
| `numpy` | `np.random` -> simulation |

### Key syntax

```python
# --- Empirical probability = count / total ---
P_diabetic = (df['Outcome'] == 1).mean()       # proportion of 1s
P_high_gluc = (df['Glucose'] > 140).mean()

# --- Define events as boolean Series ---
A = df['Glucose'] > 140
B = df['BMI'] > 30

# --- Set operations ---
(A & B).mean()    # P(A ∩ B)  -> AND / intersection
(A | B).mean()    # P(A ∪ B)  -> OR  / union

# --- Addition rule ---
P_A_or_B = A.mean() + B.mean() - (A & B).mean()

# --- Conditional probability: P(Diabetic | High Glucose) ---
high_glucose = df[df['Glucose'] > 140]          # filter to subgroup
P_D_given_HG = (high_glucose['Outcome'] == 1).mean()

# --- Contingency table ---
pd.crosstab(df['Glucose'] > 140, df['Outcome'],
            margins=True, normalize='all').round(3)

# --- Simulation (with replacement = independent) ---
rng = np.random.default_rng(seed=42)
draws = rng.choice(df['Outcome'].values, size=2, replace=True)

# --- Simulation (without replacement = dependent) ---
draws = rng.choice(df['Outcome'].values, size=2, replace=False)
```

> **Key distinction:** 
- `replace=True` → events are **independent**. 
- `replace=False` → events are **dependent** (probabilities change after each draw).

---

## Module D: Distributions & Skewness (Student 1 and Student 2)
*Tasks 8 – 9*

### Core libraries
| Library | What you use it for |
|---|---|
| `numpy.random` | Generating synthetic distributions |
| `scipy.stats` | Skewness, normality tests, distribution fitting |
| `matplotlib` | Multi-panel plots |

### Key syntax

```python
rng = np.random.default_rng(seed=42)

# --- Generate the four distributions ---
normal_data  = rng.normal(loc=0,  scale=1,  size=1000)
binom_data   = rng.binomial(n=10, p=0.35,   size=1000)
poisson_data = rng.poisson(lam=3,            size=1000)
exp_data     = rng.exponential(scale=2,      size=1000)

# --- Multi-panel plot ---
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, data, title in zip(axes.flat, [normal_data, binom_data, poisson_data, exp_data],
                            ['Normal', 'Binomial', 'Poisson', 'Exponential']):
    ax.hist(data, bins=30, edgecolor='white')
    ax.set_title(title)
plt.tight_layout(); plt.show()

# --- Skewness & mean–median gap ---
df['Insulin'].skew()
df['Insulin'].mean() - df['Insulin'].median()   # positive → right skew

# --- Log-transform to reduce right skew ---
insulin_log = np.log1p(df['Insulin'])           # log(x + 1) -> safe with zeros
insulin_log.skew()                              # should be much closer to 0

# --- Normality test ---
stat, p = stats.shapiro(df['Glucose'].dropna().sample(500, random_state=42))
print(f"p = {p:.4f} → {'Normal' if p > 0.05 else 'Not normal'}")
```

> **Log-transform rule:**
-  If skewness > 1, try `np.log1p()`. 
- If mean >> median (>> means "much greater than"), the distribution is positively skewed (right-skewed),  median is the better summary.

---



## Quick Lookup Table

| What you want | How to write it |
|---|---|
| Load CSV | `pd.read_csv("file.csv")` |
| First look | `df.head()` · `df.shape` · `df.dtypes` |
| Summary stats | `df.describe()` |
| Count per category | `df['col'].value_counts()` |
| Replace a value | `df['col'].replace(0, np.nan)` |
| Drop NaN rows | `df.dropna()` |
| Fill NaN | `df.fillna(df.median())` |
| Filter rows | `df[df['Glucose'] > 140]` |
| Group means | `df.groupby('Outcome').mean()` |
| Probability (any event) | `(df['col'] > x).mean()` |
| Intersection P(A∩B) | `(A & B).mean()` |
| Union P(A∪B) | `(A \| B).mean()` |
| Skewness | `df['col'].skew()` |
| Log transform | `np.log1p(df['col'])` |
| Histogram | `df['col'].plot.hist(bins=30)` |
| Boxplot by group | `sns.boxplot(data=df, x='Outcome', y='col')` |
| Vertical line on plot | `plt.axvline(value, color='red')` |
| Show plot | `plt.show()` |

---